# MentalDistress Phase 10 — Controls and Allocation Analysis (GPU)

Phase 10 evaluates whether confusion-targeted pair specialists provide evidence beyond a
flat classifier, indiscriminate routing, generic matched capacity, and alternative pair
allocations. It uses locked training and validation data only. The test split remains
unopened.

## Runtime and phase boundary

The notebook processes one model seed at a time on a Kaggle GPU. It freezes Phase 5 `H`,
all Phase 8 heads, and the Phase 9 selection `tau=0.10`. It does not perform Phase 11
analyses or any locked-test evaluation.

In [1]:
import copy
import gc
import hashlib
import json
import os
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from safetensors.torch import load_file
from safetensors.torch import save_file
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_DIR = WORKING_ROOT / "phase10_controls_allocation_results"
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running Phase 10.")

DEVICE = torch.device("cuda")
LABELS = ("Suicidal", "Depressed", "Anxious", "Frustrated", "Others")
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}
SEEDS = (13, 42, 77, 123, 211, 340, 512, 1024, 2026, 3407)
HARD_PAIRS = (
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others",
)
K_FIXED = 3
FROZEN_TAU = 0.10
TAU_GRID = np.array(
    [0.00, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.70, 0.90, 1.00]
)
ALLOCATION_RNG_SEED = 20260822
TRAIN_SHA256 = "1bac0d2f4e0f4d3da940b08a21bd3c6e18b1d958c57cb263c89a900381572d5b"
VALIDATION_SHA256 = "88835cd714b02efbd230536c7de3f455e807051a23a1786640b0e8b26b430165"

PHASE5_DIRNAME = "phase5_bootstrap_stability_results"
PHASE8_DIRNAME = "phase8_all_pair_specialists_results"
PHASE9_DIRNAME = "phase9_routing_calibration_results"
ACCESSED_INPUTS = []

print("CUDA device:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Frozen tau:", FROZEN_TAU)
print("Test data accessed: False")

CUDA device: Tesla T4
PyTorch: 2.10.0+cu128
Frozen tau: 0.1
Test data accessed: False


## Required Kaggle inputs

Attach the locked train and validation CSVs, Phase 5 results, all Phase 8 heads, the
executed Phase 9 results, the ten canonical MentalRoBERTa runs, and their ten-seed summary.
Inputs may be attached as result directories or deterministic ZIP archives. Every manifest
is verified before inference.

In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member_name in archive.namelist():
            member_path = (destination / member_name).resolve()
            if not str(member_path).startswith(str(destination) + os.sep):
                raise ValueError(f"Unsafe ZIP member: {member_name}")
        archive.extractall(destination)


def verify_manifest(root):
    manifest_path = root / "sha256_manifest.csv"
    frame = pd.read_csv(manifest_path)
    actual = {
        path.relative_to(root).as_posix()
        for path in root.rglob("*")
        if path.is_file() and path != manifest_path
    }
    if set(frame["path"]) != actual:
        raise ValueError(f"Manifest membership mismatch: {root}")
    for row in frame.itertuples():
        path = root / row.path
        if not path.is_file() or path.stat().st_size != int(row.bytes):
            raise ValueError(f"Manifest byte mismatch: {row.path}")
        if sha256_file(path) != row.sha256:
            raise ValueError(f"Manifest hash mismatch: {row.path}")
    return frame


def find_input(filename, role):
    matches = sorted(INPUT_ROOT.rglob(filename), key=str)
    if not matches:
        raise FileNotFoundError(f"Required Kaggle input not found: {filename}")
    path = matches[0]
    ACCESSED_INPUTS.append(
        {
            "role": role,
            "path": str(path),
            "sha256": sha256_file(path),
            "bytes": path.stat().st_size,
        }
    )
    return path


def resolve_result_dir(directory_name, archive_name, cache_name):
    directories = sorted(
        [path for path in INPUT_ROOT.rglob(directory_name) if path.is_dir()],
        key=str,
    )
    if directories:
        result_dir = directories[0]
    else:
        archives = sorted(INPUT_ROOT.rglob(archive_name), key=str)
        if not archives:
            raise FileNotFoundError(
                f"Attach {archive_name} or the extracted {directory_name} directory"
            )
        cache = WORKING_ROOT / cache_name
        if cache.exists():
            shutil.rmtree(cache)
        cache.mkdir(parents=True)
        safe_extract(archives[0], cache)
        result_dir = cache / directory_name
        ACCESSED_INPUTS.append(
            {
                "role": directory_name + "_archive",
                "path": str(archives[0]),
                "sha256": sha256_file(archives[0]),
                "bytes": archives[0].stat().st_size,
            }
        )
    verify_manifest(result_dir)
    manifest_path = result_dir / "sha256_manifest.csv"
    ACCESSED_INPUTS.append(
        {
            "role": directory_name + "_manifest",
            "path": str(manifest_path),
            "sha256": sha256_file(manifest_path),
            "bytes": manifest_path.stat().st_size,
        }
    )
    return result_dir


def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n")


def pair_slug(pair_name):
    first_label, second_label = pair_name.split("|")
    return first_label.lower() + "__" + second_label.lower()


print("PASS: path-safe discovery and integrity helpers initialized")

PASS: path-safe discovery and integrity helpers initialized


## Frozen controls

The next cell is the locally tested deterministic core embedded directly into this
notebook. It defines pair enumeration, allocation freezing, residual parameter matching,
routing, metrics, correction/harm accounting, checkpoint selection, and allocation ranking.

In [3]:
import itertools
import random

import numpy as np


def generate_pairs(labels):
    return tuple("|".join(pair) for pair in itertools.combinations(labels, 2))


def enumerate_allocations(pairs, k):
    return tuple(itertools.combinations(tuple(pairs), k))


def allocation_id(allocation, pair_order):
    order = {pair: index for index, pair in enumerate(pair_order)}
    canonical = sorted(tuple(allocation), key=order.__getitem__)
    return ";".join(canonical)


def choose_random_nonhard(pairs, hard_pairs, k, seed):
    hard_set = set(hard_pairs)
    nonhard_pairs = tuple(pair for pair in pairs if pair not in hard_set)
    candidates = enumerate_allocations(nonhard_pairs, k)
    if not candidates:
        raise ValueError("No valid non-hard allocation candidates")
    return random.Random(seed).choice(candidates)


def choose_lowest_bcr(pair_decisions, k, pair_order):
    order = {pair: index for index, pair in enumerate(pair_order)}
    ranked = sorted(
        pair_decisions,
        key=lambda row: (float(row["mean_bcr"]), order[row["pair"]]),
    )
    return tuple(row["pair"] for row in ranked[:k])


def match_residual_width(feature_dim, k, class_count=5):
    ctpr_parameters = k * (2 * feature_dim + 2)
    candidates = []
    for hidden_width in range(1, feature_dim + 1):
        residual_parameters = hidden_width * (feature_dim + 1)
        residual_parameters += class_count * (hidden_width + 1)
        candidates.append(
            (
                abs(residual_parameters - ctpr_parameters),
                hidden_width,
                residual_parameters,
            )
        )
    absolute_difference, hidden_width, residual_parameters = min(candidates)
    return {
        "feature_dim": feature_dim,
        "k": k,
        "class_count": class_count,
        "ctpr_parameters": ctpr_parameters,
        "hidden_width": hidden_width,
        "residual_parameters": residual_parameters,
        "absolute_difference": absolute_difference,
    }


def route_allocation(
    top2,
    margin,
    base,
    specialist_predictions,
    allocation,
    tau,
    label_to_id,
):
    top2 = np.asarray(top2)
    margin = np.asarray(margin)
    base = np.asarray(base)
    if top2.shape != (len(base), 2) or margin.shape != (len(base),):
        raise ValueError("Routing inputs have incompatible shapes")

    final = base.copy()
    routed = np.zeros(len(base), dtype=bool)
    routed_pair = np.full(len(base), "", dtype=object)
    if np.isclose(float(tau), 0.0):
        return final, routed, routed_pair

    top2_sets = tuple(frozenset(row) for row in top2)
    for pair_name in allocation:
        if pair_name not in specialist_predictions:
            raise KeyError(f"Missing specialist predictions for {pair_name}")
        first_label, second_label = pair_name.split("|")
        pair_ids = frozenset((label_to_id[first_label], label_to_id[second_label]))
        pair_matches = np.fromiter(
            (row_set == pair_ids for row_set in top2_sets),
            dtype=bool,
            count=len(base),
        )
        gate = pair_matches & (margin <= float(tau))
        predictions = np.asarray(specialist_predictions[pair_name])
        if predictions.shape != base.shape:
            raise ValueError(f"Specialist prediction shape mismatch for {pair_name}")
        final[gate] = predictions[gate]
        routed |= gate
        routed_pair[gate] = pair_name
    return final, routed, routed_pair


def confusion_matrix_array(y_true, y_predicted, class_count=5):
    y_true = np.asarray(y_true, dtype=int)
    y_predicted = np.asarray(y_predicted, dtype=int)
    if y_true.shape != y_predicted.shape:
        raise ValueError("Metric inputs have incompatible shapes")
    if ((y_true < 0) | (y_true >= class_count)).any():
        raise ValueError("True labels are outside the class range")
    if ((y_predicted < 0) | (y_predicted >= class_count)).any():
        raise ValueError("Predictions are outside the class range")
    encoded = class_count * y_true + y_predicted
    return np.bincount(encoded, minlength=class_count**2).reshape(
        class_count,
        class_count,
    )


def classification_metrics(y_true, y_predicted, class_count=5):
    confusion = confusion_matrix_array(y_true, y_predicted, class_count)
    true_positive = np.diag(confusion).astype(float)
    support = confusion.sum(axis=1).astype(float)
    predicted = confusion.sum(axis=0).astype(float)
    recall = np.divide(
        true_positive,
        support,
        out=np.zeros(class_count, dtype=float),
        where=support != 0,
    )
    precision = np.divide(
        true_positive,
        predicted,
        out=np.zeros(class_count, dtype=float),
        where=predicted != 0,
    )
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros(class_count, dtype=float),
        where=(precision + recall) != 0,
    )
    return {
        "accuracy": float(true_positive.sum() / confusion.sum()),
        "macro_f1": float(f1.mean()),
        "balanced_accuracy": float(recall.mean()),
    }


def correction_harm_counts(y_true, base, final, routed):
    y_true = np.asarray(y_true)
    base = np.asarray(base)
    final = np.asarray(final)
    routed = np.asarray(routed, dtype=bool)
    if not (y_true.shape == base.shape == final.shape == routed.shape):
        raise ValueError("Correction/harm inputs have incompatible shapes")
    corrections = int((routed & (base != y_true) & (final == y_true)).sum())
    harms = int((routed & (base == y_true) & (final != y_true)).sum())
    return {
        "corrections": corrections,
        "harms": harms,
        "net_corrections": corrections - harms,
    }


def choose_best_checkpoint(records):
    if not records:
        raise ValueError("Checkpoint records are empty")
    return min(
        range(len(records)),
        key=lambda index: (
            -float(records[index]["macro_f1"]),
            float(records[index]["validation_loss"]),
            int(records[index]["epoch"]),
        ),
    )


def rank_allocation_records(records, hard_allocation_id):
    ranked = sorted(
        (dict(record) for record in records),
        key=lambda row: (
            -float(row["mean_macro_f1"]),
            float(row["mean_coverage"]),
            row["allocation_id"],
        ),
    )
    denominator = max(len(ranked) - 1, 1)
    for index, row in enumerate(ranked, start=1):
        row["rank"] = index
        row["percentile"] = 100.0 * (len(ranked) - index) / denominator
        row["is_hard_allocation"] = row["allocation_id"] == hard_allocation_id
    return ranked

## Lightweight protocol self-tests

These tests run before any dataset or model is loaded. They prove the 120-allocation space,
the frozen random allocation, the matched residual width, and the zero-routing control.

In [4]:
ALL_PAIRS = generate_pairs(LABELS)
ALL_ALLOCATIONS = enumerate_allocations(ALL_PAIRS, K_FIXED)
RANDOM_NONHARD = choose_random_nonhard(
    ALL_PAIRS,
    HARD_PAIRS,
    K_FIXED,
    ALLOCATION_RNG_SEED,
)
EXPECTED_RANDOM_NONHARD = (
    "Depressed|Anxious",
    "Anxious|Frustrated",
    "Anxious|Others",
)
assert len(ALL_PAIRS) == 10
assert len(ALL_ALLOCATIONS) == 120
assert RANDOM_NONHARD == EXPECTED_RANDOM_NONHARD

width_test = match_residual_width(768, K_FIXED, len(LABELS))
assert width_test["ctpr_parameters"] == 4614
assert width_test["hidden_width"] == 6
assert width_test["residual_parameters"] == 4649

toy_top2 = np.array([[0, 1], [3, 4]])
toy_margin = np.zeros(2)
toy_base = toy_top2[:, 0]
toy_specialists = {
    "Suicidal|Depressed": np.array([1, 0]),
    "Frustrated|Others": np.array([0, 4]),
    "Depressed|Others": np.array([0, 0]),
}
toy_final, toy_routed, _ = route_allocation(
    toy_top2,
    toy_margin,
    toy_base,
    toy_specialists,
    HARD_PAIRS,
    0.00,
    LABEL_TO_ID,
)
assert np.array_equal(toy_final, toy_base)
assert not toy_routed.any()
print("PASS: lightweight Phase 10 helper tests")

PASS: lightweight Phase 10 helper tests


## Verify locked data and Phase 5

This gate verifies exact dataset hashes and freezes `H`, `K`, the random non-hard
allocation, and the three lowest-BCR pairs before any model output is evaluated.

In [5]:
train_path = find_input("MentalDistress_train_LOCKED.csv", "locked_train")
validation_path = find_input(
    "MentalDistress_validation_LOCKED.csv",
    "locked_validation",
)
if sha256_file(train_path) != TRAIN_SHA256:
    raise ValueError("Locked training SHA-256 mismatch")
if sha256_file(validation_path) != VALIDATION_SHA256:
    raise ValueError("Locked validation SHA-256 mismatch")

train = pd.read_csv(train_path)
validation = pd.read_csv(validation_path)
assert len(train) == 6954
assert len(validation) == 1490
for frame in (train, validation):
    assert {"original_row", "text", "label"}.issubset(frame.columns)
    assert set(frame["label"]) == set(LABELS)
    assert frame["original_row"].is_unique

phase5_dir = resolve_result_dir(
    PHASE5_DIRNAME,
    "MentalDistress_Phase5_Bootstrap_Stability_Results.zip",
    "phase10_phase5_input",
)
phase5_selection = json.loads(
    (phase5_dir / "phase5_hard_pair_selection.json").read_text()
)
assert phase5_selection["status"] == "FROZEN"
assert phase5_selection["H"] == list(HARD_PAIRS)
assert phase5_selection["K"] == K_FIXED
assert phase5_selection["test_accessed"] is False

LOWEST_BCR = choose_lowest_bcr(
    phase5_selection["pair_decisions"],
    K_FIXED,
    ALL_PAIRS,
)
EXPECTED_LOWEST_BCR = {
    "Depressed|Frustrated",
    "Suicidal|Anxious",
    "Anxious|Frustrated",
}
assert set(LOWEST_BCR) == EXPECTED_LOWEST_BCR
assert set(RANDOM_NONHARD).isdisjoint(HARD_PAIRS)

y_train = train["label"].map(LABEL_TO_ID).to_numpy(dtype=int)
y_validation = validation["label"].map(LABEL_TO_ID).to_numpy(dtype=int)
print("Train rows:", len(train), "Validation rows:", len(validation))
print("Frozen H:", HARD_PAIRS)
print("Random non-hard:", RANDOM_NONHARD)
print("Lowest BCR:", LOWEST_BCR)
print("PASS: locked data and Phase 5 controls verified")

Train rows: 6954 Validation rows: 1490
Frozen H: ('Frustrated|Others', 'Suicidal|Depressed', 'Depressed|Others')
Random non-hard: ('Depressed|Anxious', 'Anxious|Frustrated', 'Anxious|Others')
Lowest BCR: ('Anxious|Frustrated', 'Suicidal|Anxious', 'Depressed|Frustrated')
PASS: locked data and Phase 5 controls verified


## Verify Phase 8 and executed Phase 9

Phase 8 must contain all 100 frozen heads. Phase 9 must be the executed archive audited for
this project, with `selected_tau=0.10`, specialist routing selected, a true no-routing
baseline, and no test access.

In [6]:
phase8_dir = resolve_result_dir(
    PHASE8_DIRNAME,
    "MentalDistress_Phase8_All_Pair_Specialists_Results.zip",
    "phase10_phase8_input",
)
phase8_protocol = json.loads((phase8_dir / "phase8_protocol.json").read_text())
assert phase8_protocol["phase"] == 8
assert phase8_protocol["head_count"] == 100
assert phase8_protocol["model_seeds"] == list(SEEDS)
assert phase8_protocol["pairs"] == list(ALL_PAIRS)
assert phase8_protocol["phase5_frozen_H"] == list(HARD_PAIRS)
assert phase8_protocol["phase5_frozen_K"] == K_FIXED
assert phase8_protocol["routing_performed"] is False
assert phase8_protocol["test_accessed"] is False

for model_seed in SEEDS:
    for pair_name in ALL_PAIRS:
        head_dir = (
            phase8_dir
            / "heads"
            / f"seed_{model_seed}"
            / pair_slug(pair_name)
        )
        assert (head_dir / "head.safetensors").is_file()
        assert (head_dir / "metadata.json").is_file()

phase9_dir = resolve_result_dir(
    PHASE9_DIRNAME,
    "MentalDistress_Phase9_Routing_Calibration_Results.zip",
    "phase10_phase9_input",
)
phase9_config = json.loads(
    (phase9_dir / "phase9_selected_config.json").read_text()
)
assert phase9_config["phase"] == 9
assert phase9_config["K_fixed"] == K_FIXED
assert phase9_config["H"] == list(HARD_PAIRS)
assert np.allclose(phase9_config["tau_grid"], TAU_GRID)
assert phase9_config["no_routing_baseline_included"] is True
assert np.isclose(phase9_config["selected_tau"], FROZEN_TAU)
assert phase9_config["selected_mode"] == "specialist_routing"
assert phase9_config["test_accessed"] is False

phase9_grid = pd.read_csv(phase9_dir / "phase9_routing_grid.csv")
phase9_seedwise = pd.read_csv(phase9_dir / "phase9_routing_seedwise.csv")
assert len(phase9_grid) == len(TAU_GRID)
assert len(phase9_seedwise) == len(SEEDS) * len(TAU_GRID)
assert np.allclose(phase9_grid["tau"], TAU_GRID)
eligible = phase9_grid[
    phase9_grid["mean_macro_f1"]
    >= phase9_grid["mean_macro_f1"].max() - 0.001
]
reproduced_tau = float(
    eligible.sort_values(["mean_coverage", "tau"]).iloc[0]["tau"]
)
assert np.isclose(reproduced_tau, FROZEN_TAU)
print("PASS: Phase 8 bank and frozen Phase 9 configuration verified")

PASS: Phase 8 bank and frozen Phase 9 configuration verified


## Verify canonical global checkpoints

The ten-seed summary resolves one complete model/artifact run per principal seed. Model and
validation-artifact manifests are verified before any checkpoint reaches the GPU.

In [7]:
global_summary_path = find_input(
    "mentalroberta_full_10seed_summary.csv",
    "global_summary",
)
global_summary = pd.read_csv(global_summary_path)
assert sorted(global_summary["seed"].astype(int).tolist()) == list(SEEDS)

global_runs = {}
for summary_row in global_summary.itertuples():
    model_seed = int(summary_row.seed)
    candidates = [
        path
        for path in INPUT_ROOT.rglob(str(summary_row.run_id))
        if path.is_dir()
        and (path / "models/best_model/model.safetensors").is_file()
        and (path / "artifacts/validation_predictions.csv").is_file()
    ]
    if not candidates:
        raise FileNotFoundError(f"Complete global run missing: {summary_row.run_id}")
    run_dir = sorted(candidates, key=str)[0]
    verify_manifest(run_dir / "models")
    verify_manifest(run_dir / "artifacts")
    global_runs[model_seed] = run_dir
    for role, manifest_path in (
        ("global_model_manifest", run_dir / "models/sha256_manifest.csv"),
        ("global_artifact_manifest", run_dir / "artifacts/sha256_manifest.csv"),
    ):
        ACCESSED_INPUTS.append(
            {
                "role": role,
                "model_seed": model_seed,
                "path": str(manifest_path),
                "sha256": sha256_file(manifest_path),
                "bytes": manifest_path.stat().st_size,
            }
        )
    print(f"  seed {model_seed}: {run_dir.name}")

assert sorted(global_runs) == list(SEEDS)
print("PASS: ten canonical global checkpoints verified")

  seed 13: mentalroberta_full_seed13_len512_fea2991778f3
  seed 42: mentalroberta_full_seed42_len512_be72f916ca15
  seed 77: mentalroberta_full_seed77_len512_d14375990239
  seed 123: mentalroberta_full_seed123_len512_d1ad313ee75d
  seed 211: mentalroberta_full_seed211_len512_149308113b40
  seed 340: mentalroberta_full_seed340_len512_6353292c8230
  seed 512: mentalroberta_full_seed512_len512_412e1caa7392
  seed 1024: mentalroberta_full_seed1024_len512_e91f034d8a71
  seed 2026: mentalroberta_full_seed2026_len512_f6630d942394
  seed 3407: mentalroberta_full_seed3407_len512_d8596ea710c3
PASS: ten canonical global checkpoints verified


## Frozen model and specialist inference

Each seed's global model generates frozen train/validation token-zero representations and
logits. All ten matching Phase 8 heads are then evaluated on every validation row. No
encoder or specialist parameter receives a gradient.

In [8]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)


def encode_frame(tokenizer, model, frame, batch_size=16):
    feature_batches = []
    logit_batches = []
    texts = frame["text"].astype(str).tolist()
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            encoded = tokenizer(
                texts[start : start + batch_size],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            encoded = {name: tensor.to(DEVICE) for name, tensor in encoded.items()}
            output = model(
                **encoded,
                output_hidden_states=True,
                return_dict=True,
            )
            feature_batches.append(
                output.hidden_states[-1][:, 0, :].float().cpu()
            )
            logit_batches.append(output.logits.float().cpu())
    return torch.cat(feature_batches), torch.cat(logit_batches)


def extract_seed_outputs(run_dir):
    model_dir = run_dir / "models/best_model"
    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        local_files_only=True,
        output_hidden_states=True,
    )
    assert model.config.num_labels == len(LABELS)
    model = model.eval().to(DEVICE)
    train_features, train_logits = encode_frame(tokenizer, model, train)
    validation_features, validation_logits = encode_frame(
        tokenizer,
        model,
        validation,
    )
    del model, tokenizer
    torch.cuda.empty_cache()
    return train_features, train_logits, validation_features, validation_logits


def load_all_specialist_predictions(model_seed, validation_features):
    predictions = {}
    feature_array = validation_features.numpy()
    for pair_name in ALL_PAIRS:
        head_dir = (
            phase8_dir
            / "heads"
            / f"seed_{model_seed}"
            / pair_slug(pair_name)
        )
        metadata = json.loads((head_dir / "metadata.json").read_text())
        assert metadata["phase"] == 8
        assert metadata["model_seed"] == model_seed
        assert metadata["pair"] == pair_name
        assert metadata["status"] == "PASS"
        assert metadata["test_accessed"] is False
        tensors = load_file(str(head_dir / "head.safetensors"), device="cpu")
        weight = tensors["classifier.weight"].numpy()
        bias = tensors["classifier.bias"].numpy()
        assert weight.shape == (2, validation_features.shape[1])
        head_logits = feature_array @ weight.T + bias
        first_label, second_label = pair_name.split("|")
        predictions[pair_name] = np.where(
            head_logits.argmax(axis=1) == 0,
            LABEL_TO_ID[first_label],
            LABEL_TO_ID[second_label],
        )
    return predictions


print("PASS: frozen encoder and specialist helpers initialized")

PASS: frozen encoder and specialist helpers initialized


## Parameter-matched residual control

A3 adds a globally applied `d → h → 5` GELU residual MLP to frozen global logits. The final
layer starts at zero, so epoch zero exactly equals A0. Only this residual module is trained.
Width is selected solely by parameter-count proximity.

In [9]:
class ParameterMatchedResidual(torch.nn.Module):
    def __init__(self, feature_dim, hidden_width, class_count):
        super().__init__()
        self.first = torch.nn.Linear(feature_dim, hidden_width)
        self.activation = torch.nn.GELU()
        self.final = torch.nn.Linear(hidden_width, class_count)
        torch.nn.init.zeros_(self.final.weight)
        torch.nn.init.zeros_(self.final.bias)

    def forward(self, features, global_logits):
        residual_logits = self.final(self.activation(self.first(features)))
        return global_logits + residual_logits


def residual_class_weights(labels):
    counts = np.bincount(labels, minlength=len(LABELS)).astype(float)
    weights = len(labels) / (len(LABELS) * counts)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def evaluate_residual(model, features, global_logits, labels, criterion):
    model.eval()
    predictions = []
    total_loss = 0.0
    with torch.inference_mode():
        for start in range(0, len(labels), 256):
            stop = start + 256
            batch_features = features[start:stop].to(DEVICE)
            batch_logits = global_logits[start:stop].to(DEVICE)
            batch_labels = torch.tensor(
                labels[start:stop],
                dtype=torch.long,
                device=DEVICE,
            )
            output = model(batch_features, batch_logits)
            total_loss += criterion(output, batch_labels).item() * len(batch_labels)
            predictions.append(output.argmax(dim=1).cpu().numpy())
    prediction = np.concatenate(predictions)
    return prediction, total_loss / len(labels)


def train_residual_control(
    model_seed,
    train_features,
    train_logits,
    validation_features,
    validation_logits,
):
    residual_seed = 100000 + model_seed
    seed_everything(residual_seed)
    parameter_spec = match_residual_width(
        train_features.shape[1],
        K_FIXED,
        len(LABELS),
    )
    assert parameter_spec["hidden_width"] == 6
    assert parameter_spec["ctpr_parameters"] == 4614
    assert parameter_spec["residual_parameters"] == 4649

    model = ParameterMatchedResidual(
        parameter_spec["feature_dim"],
        parameter_spec["hidden_width"],
        parameter_spec["class_count"],
    ).to(DEVICE)
    criterion = torch.nn.CrossEntropyLoss(
        weight=residual_class_weights(y_train)
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-2,
    )
    generator = torch.Generator().manual_seed(residual_seed)
    dataset = torch.utils.data.TensorDataset(
        train_features,
        train_logits,
        torch.tensor(y_train, dtype=torch.long),
    )
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=64,
        shuffle=True,
        generator=generator,
    )

    epoch_zero_prediction, epoch_zero_loss = evaluate_residual(
        model,
        validation_features,
        validation_logits,
        y_validation,
        criterion,
    )
    global_prediction = validation_logits.argmax(dim=1).numpy()
    assert np.array_equal(epoch_zero_prediction, global_prediction)

    history = []
    best_state = None
    best_history_index = None
    stale_epochs = 0
    for epoch in range(1, 61):
        model.train()
        train_loss_sum = 0.0
        for batch_features, batch_logits, batch_labels in loader:
            batch_features = batch_features.to(DEVICE)
            batch_logits = batch_logits.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            output = model(batch_features, batch_logits)
            loss = criterion(output, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss_sum += loss.item() * len(batch_labels)

        prediction, validation_loss = evaluate_residual(
            model,
            validation_features,
            validation_logits,
            y_validation,
            criterion,
        )
        validation_metrics = classification_metrics(
            y_validation,
            prediction,
            len(LABELS),
        )
        history.append(
            {
                "model_seed": model_seed,
                "residual_seed": residual_seed,
                "epoch": epoch,
                "train_loss": train_loss_sum / len(y_train),
                "validation_loss": validation_loss,
                **validation_metrics,
            }
        )
        selected_index = choose_best_checkpoint(history)
        if selected_index != best_history_index:
            best_history_index = selected_index
            if selected_index == len(history) - 1:
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }
                stale_epochs = 0
            else:
                stale_epochs += 1
        else:
            stale_epochs += 1
        if stale_epochs >= 6:
            break

    if best_state is None:
        raise RuntimeError("Residual training did not produce a checkpoint")
    model.load_state_dict(best_state)
    final_prediction, final_loss = evaluate_residual(
        model,
        validation_features,
        validation_logits,
        y_validation,
        criterion,
    )
    best_record = history[best_history_index]
    assert np.isclose(final_loss, best_record["validation_loss"], atol=1e-7)
    return {
        "model": model,
        "state": best_state,
        "prediction": final_prediction,
        "history": history,
        "best_record": best_record,
        "epoch_zero_loss": epoch_zero_loss,
        "parameter_spec": parameter_spec,
        "residual_seed": residual_seed,
    }


print("PASS: parameter-matched residual training helpers initialized")

PASS: parameter-matched residual training helpers initialized


## Phase 9 reproduction gate

Before Phase 10 controls are accepted, the notebook must reproduce the selected Phase 9
predictions for all ten seeds. Global and final IDs, routing flags, and pair identities are
exact; margins use absolute tolerance `1e-6`.

In [10]:
CONTROL_IDS = (
    "A0_flat",
    "A1_conditional",
    "A2_always_route",
    "A3_parameter_matched_residual",
    "A4_random_nonhard",
    "A6_lowest_bcr",
)
A5_MARKER = "A5_exhaustive_allocation"
A7_MARKER = "A7_threshold_sensitivity"


def control_result_row(
    model_seed,
    control_id,
    base,
    final,
    active,
    tau,
    allocation,
):
    result = classification_metrics(y_validation, final, len(LABELS))
    counts = correction_harm_counts(y_validation, base, final, active)
    result.update(counts)
    result.update(
        {
            "model_seed": model_seed,
            "control_id": control_id,
            "tau": tau,
            "allocation_id": (
                allocation_id(allocation, ALL_PAIRS) if allocation else ""
            ),
            "routing_coverage": float(np.mean(active)),
            "routed_n": int(np.sum(active)),
        }
    )
    return result


def save_control_artifacts(
    model_seed,
    control_id,
    base,
    final,
    active,
    routed_pair,
    margin,
    top2,
):
    control_dir = OUTPUT_DIR / "controls" / control_id / f"seed_{model_seed}"
    control_dir.mkdir(parents=True, exist_ok=True)
    prediction_frame = pd.DataFrame(
        {
            "original_row": validation["original_row"],
            "true_label": validation["label"],
            "global_id": base,
            "final_id": final,
            "active": active,
            "routed_pair": routed_pair,
            "margin": margin,
            "top1": top2[:, 0],
            "top2": top2[:, 1],
        }
    )
    prediction_frame.to_csv(control_dir / "validation_predictions.csv", index=False)
    confusion = confusion_matrix_array(
        y_validation,
        final,
        len(LABELS),
    )
    pd.DataFrame(confusion, index=LABELS, columns=LABELS).to_csv(
        control_dir / "confusion_matrix.csv"
    )
    report = classification_report(
        y_validation,
        final,
        labels=range(len(LABELS)),
        target_names=LABELS,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).T.to_csv(control_dir / "classification_report.csv")


control_rows = []
allocation_rows = []
reproduction_rows = []
residual_metric_rows = []
residual_history_rows = []
print("PASS: control, reproduction, and export helpers initialized")

PASS: control, reproduction, and export helpers initialized


## Execute ten seed-matched control experiments

For each seed, the notebook extracts frozen outputs, reproduces Phase 9, trains A3, saves
six named control systems, and evaluates all 120 allocations at `tau=0.10`. Only one global
model and one seed's representations remain resident at a time.

In [11]:
hard_allocation_id = allocation_id(HARD_PAIRS, ALL_PAIRS)
random_allocation_id = allocation_id(RANDOM_NONHARD, ALL_PAIRS)
lowest_bcr_allocation_id = allocation_id(LOWEST_BCR, ALL_PAIRS)

for seed_index, model_seed in enumerate(SEEDS, start=1):
    print(f"[{seed_index}/{len(SEEDS)}] seed {model_seed}")
    (
        train_features,
        train_logits,
        validation_features,
        validation_logits,
    ) = extract_seed_outputs(global_runs[model_seed])
    assert train_features.shape[0] == len(train)
    assert validation_features.shape[0] == len(validation)
    assert train_features.shape[1] == validation_features.shape[1] == 768

    probabilities = torch.softmax(validation_logits, dim=1).numpy()
    top2 = np.argsort(-probabilities, axis=1)[:, :2]
    row_indices = np.arange(len(validation))
    margin = (
        probabilities[row_indices, top2[:, 0]]
        - probabilities[row_indices, top2[:, 1]]
    )
    base = top2[:, 0]
    specialist_predictions = load_all_specialist_predictions(
        model_seed,
        validation_features,
    )

    a1_final, a1_routed, a1_pair = route_allocation(
        top2,
        margin,
        base,
        specialist_predictions,
        HARD_PAIRS,
        FROZEN_TAU,
        LABEL_TO_ID,
    )
    saved_phase9 = pd.read_csv(
        phase9_dir / f"selected_routing_predictions_seed{model_seed}.csv"
    )
    assert np.array_equal(
        saved_phase9["original_row"].to_numpy(),
        validation["original_row"].to_numpy(),
    )
    assert np.array_equal(saved_phase9["global_id"].to_numpy(), base)
    assert np.array_equal(saved_phase9["final_id"].to_numpy(), a1_final)
    assert np.array_equal(saved_phase9["routed"].to_numpy(), a1_routed)
    saved_pair = saved_phase9["routed_pair"].fillna("").to_numpy()
    assert np.array_equal(saved_pair, a1_pair)
    assert np.array_equal(saved_phase9["top1"].to_numpy(), top2[:, 0])
    assert np.array_equal(saved_phase9["top2"].to_numpy(), top2[:, 1])
    margin_max_abs_diff = float(
        np.max(np.abs(saved_phase9["margin"].to_numpy() - margin))
    )
    assert margin_max_abs_diff <= 1e-6
    reproduction_rows.append(
        {
            "model_seed": model_seed,
            "selected_tau": FROZEN_TAU,
            "prediction_exact": True,
            "routing_exact": True,
            "routed_pair_exact": True,
            "margin_max_abs_diff": margin_max_abs_diff,
            "status": "PASS",
        }
    )

    residual = train_residual_control(
        model_seed,
        train_features,
        train_logits,
        validation_features,
        validation_logits,
    )
    residual_dir = OUTPUT_DIR / "residuals" / f"seed_{model_seed}"
    residual_dir.mkdir(parents=True, exist_ok=True)
    save_file(residual["state"], str(residual_dir / "residual.safetensors"))
    residual_prediction = residual["prediction"]
    residual_active = np.ones(len(validation), dtype=bool)
    residual_pair = np.full(len(validation), "", dtype=object)
    pd.DataFrame(
        {
            "original_row": validation["original_row"],
            "true_label": validation["label"],
            "global_id": base,
            "residual_id": residual_prediction,
        }
    ).to_csv(residual_dir / "validation_predictions.csv", index=False)
    residual_history_rows.extend(residual["history"])
    residual_metrics = {
        "model_seed": model_seed,
        "residual_seed": residual["residual_seed"],
        "best_epoch": residual["best_record"]["epoch"],
        "epochs_trained": len(residual["history"]),
        "epoch_zero_validation_loss": residual["epoch_zero_loss"],
        **residual["parameter_spec"],
        **classification_metrics(
            y_validation,
            residual_prediction,
            len(LABELS),
        ),
    }
    residual_metric_rows.append(residual_metrics)
    write_json(
        residual_dir / "metadata.json",
        {
            **residual_metrics,
            "architecture": "Linear(d,h)-GELU-Linear(h,5) residual logits",
            "test_accessed": False,
        },
    )

    blank_pair = np.full(len(validation), "", dtype=object)
    no_active = np.zeros(len(validation), dtype=bool)
    a2_final, a2_routed, a2_pair = route_allocation(
        top2,
        margin,
        base,
        specialist_predictions,
        HARD_PAIRS,
        1.00,
        LABEL_TO_ID,
    )
    a4_final, a4_routed, a4_pair = route_allocation(
        top2,
        margin,
        base,
        specialist_predictions,
        RANDOM_NONHARD,
        FROZEN_TAU,
        LABEL_TO_ID,
    )
    a6_final, a6_routed, a6_pair = route_allocation(
        top2,
        margin,
        base,
        specialist_predictions,
        LOWEST_BCR,
        FROZEN_TAU,
        LABEL_TO_ID,
    )
    control_payloads = (
        ("A0_flat", base, no_active, blank_pair, np.nan, ()),
        (
            "A1_conditional",
            a1_final,
            a1_routed,
            a1_pair,
            FROZEN_TAU,
            HARD_PAIRS,
        ),
        (
            "A2_always_route",
            a2_final,
            a2_routed,
            a2_pair,
            1.00,
            HARD_PAIRS,
        ),
        (
            "A3_parameter_matched_residual",
            residual_prediction,
            residual_active,
            residual_pair,
            np.nan,
            (),
        ),
        (
            "A4_random_nonhard",
            a4_final,
            a4_routed,
            a4_pair,
            FROZEN_TAU,
            RANDOM_NONHARD,
        ),
        (
            "A6_lowest_bcr",
            a6_final,
            a6_routed,
            a6_pair,
            FROZEN_TAU,
            LOWEST_BCR,
        ),
    )
    for control_id, final, active, routed_pair, tau, allocation in control_payloads:
        control_rows.append(
            control_result_row(
                model_seed,
                control_id,
                base,
                final,
                active,
                tau,
                allocation,
            )
        )
        save_control_artifacts(
            model_seed,
            control_id,
            base,
            final,
            active,
            routed_pair,
            margin,
            top2,
        )

    for allocation in ALL_ALLOCATIONS:
        final, routed, _ = route_allocation(
            top2,
            margin,
            base,
            specialist_predictions,
            allocation,
            FROZEN_TAU,
            LABEL_TO_ID,
        )
        allocation_row = control_result_row(
            model_seed,
            A5_MARKER,
            base,
            final,
            routed,
            FROZEN_TAU,
            allocation,
        )
        allocation_rows.append(allocation_row)
        if allocation_id(allocation, ALL_PAIRS) == hard_allocation_id:
            assert np.array_equal(final, a1_final)
            assert np.array_equal(routed, a1_routed)

    del residual["model"]
    del train_features, train_logits, validation_features, validation_logits
    del specialist_predictions, probabilities
    gc.collect()
    torch.cuda.empty_cache()
    print("  PASS: Phase 9 reproduced; controls and 120 allocations complete")

print("PASS: all ten Phase 10 seed experiments complete")

[1/10] seed 13


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  PASS: Phase 9 reproduced; controls and 120 allocations complete
[2/10] seed 42


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  PASS: Phase 9 reproduced; controls and 120 allocations complete
[3/10] seed 77


...

  PASS: Phase 9 reproduced; controls and 120 allocations complete
[9/10] seed 2026


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  PASS: Phase 9 reproduced; controls and 120 allocations complete
[10/10] seed 3407


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  PASS: Phase 9 reproduced; controls and 120 allocations complete
PASS: all ten Phase 10 seed experiments complete


## Exhaustive allocation analysis

All 120 size-three allocations are aggregated with equal seed weight and ranked
descriptively by mean macro-F1, lower coverage, then canonical allocation identifier. The
frozen hard allocation's rank does not replace Phase 5 or Phase 9 decisions.

In [12]:
control_seedwise = pd.DataFrame(control_rows)
allocation_seedwise = pd.DataFrame(allocation_rows)
phase9_reproduction = pd.DataFrame(reproduction_rows)
residual_metrics = pd.DataFrame(residual_metric_rows)
residual_history = pd.DataFrame(residual_history_rows)

assert len(control_seedwise) == 60
assert control_seedwise.groupby("control_id")["model_seed"].nunique().eq(10).all()
assert len(allocation_seedwise) == 1200
assert allocation_seedwise["allocation_id"].nunique() == 120
assert len(phase9_reproduction) == 10
assert phase9_reproduction["status"].eq("PASS").all()
assert len(residual_metrics) == 10

baseline_macro = (
    control_seedwise[control_seedwise["control_id"] == "A0_flat"]
    .set_index("model_seed")["macro_f1"]
)
control_seedwise["macro_f1_delta_vs_A0"] = control_seedwise.apply(
    lambda row: row["macro_f1"] - baseline_macro.loc[row["model_seed"]],
    axis=1,
)
control_summary = control_seedwise.groupby("control_id", as_index=False).agg(
    mean_macro_f1=("macro_f1", "mean"),
    sd_macro_f1=("macro_f1", "std"),
    mean_accuracy=("accuracy", "mean"),
    mean_balanced_accuracy=("balanced_accuracy", "mean"),
    mean_coverage=("routing_coverage", "mean"),
    mean_corrections=("corrections", "mean"),
    mean_harms=("harms", "mean"),
    mean_net_corrections=("net_corrections", "mean"),
    mean_macro_f1_delta_vs_A0=("macro_f1_delta_vs_A0", "mean"),
)

allocation_summary = allocation_seedwise.groupby(
    "allocation_id",
    as_index=False,
).agg(
    mean_macro_f1=("macro_f1", "mean"),
    sd_macro_f1=("macro_f1", "std"),
    mean_accuracy=("accuracy", "mean"),
    mean_balanced_accuracy=("balanced_accuracy", "mean"),
    mean_coverage=("routing_coverage", "mean"),
    mean_corrections=("corrections", "mean"),
    mean_harms=("harms", "mean"),
    mean_net_corrections=("net_corrections", "mean"),
)
ranked_records = rank_allocation_records(
    allocation_summary.to_dict("records"),
    hard_allocation_id,
)
allocation_summary = pd.DataFrame(ranked_records)
assert len(allocation_summary) == 120
assert allocation_summary["allocation_id"].nunique() == 120
hard_rank_row = allocation_summary[
    allocation_summary["allocation_id"] == hard_allocation_id
].iloc[0]

a1_by_seed = control_seedwise[
    control_seedwise["control_id"] == "A1_conditional"
].set_index("model_seed")
hard_by_seed = allocation_seedwise[
    allocation_seedwise["allocation_id"] == hard_allocation_id
].set_index("model_seed")
assert np.allclose(a1_by_seed["macro_f1"], hard_by_seed["macro_f1"])
assert np.allclose(
    a1_by_seed["routing_coverage"],
    hard_by_seed["routing_coverage"],
)
a2_by_seed = control_seedwise[
    control_seedwise["control_id"] == "A2_always_route"
].set_index("model_seed")
assert (a2_by_seed["routing_coverage"] >= a1_by_seed["routing_coverage"]).all()

threshold_sensitivity = phase9_grid.copy()
threshold_sensitivity.insert(0, "control_id", A7_MARKER)
print("Frozen hard allocation rank:", int(hard_rank_row["rank"]), "/ 120")
display(control_summary.round(6))
display(allocation_summary.head(10).round(6))
print("PASS: control aggregation and exhaustive ranking verified")

Frozen hard allocation rank: 3 / 120


,control_id,mean_macro_f1,sd_macro_f1,mean_accuracy,mean_balanced_accuracy,mean_coverage,mean_corrections,mean_harms,mean_net_corrections,mean_macro_f1_delta_vs_A0
0,A0_flat,0.863198,0.003567,0.866577,0.862223,0.000000,0.0,0.0,0.0,0.000000
1,A1_conditional,0.864138,0.004309,0.867517,0.863153,0.006577,2.6,1.2,1.4,0.000940
2,A2_always_route,0.864921,0.004024,0.868255,0.864052,0.556577,8.3,5.8,2.5,0.001723
3,A3_parameter_matched_residual,0.865044,0.003816,0.868188,0.864140,1.000000,10.2,7.8,2.4,0.001846
4,A4_random_nonhard,0.863376,0.003652,0.866711,0.862386,0.001074,0.4,0.2,0.2,0.000178
5,A6_lowest_bcr,0.863524,0.003471,0.866846,0.862550,0.001007,0.6,0.2,0.4,0.000326


,allocation_id,mean_macro_f1,sd_macro_f1,mean_accuracy,mean_balanced_accuracy,mean_coverage,mean_corrections,mean_harms,mean_net_corrections,rank,percentile,is_hard_allocation
0,Depressed|Frustrated;Depressed|Others;Frustrat...,0.864265,0.003909,0.867584,0.863283,0.005638,2.3,0.8,1.5,1,100.000000,False
1,Depressed|Anxious;Depressed|Others;Frustrated|...,0.864174,0.004136,0.867517,0.863187,0.005168,2.1,0.7,1.4,2,99.159664,False
2,Suicidal|Depressed;Depressed|Others;Frustrated...,0.864138,0.004309,0.867517,0.863153,0.006577,2.6,1.2,1.4,3,98.319328,True
3,Depressed|Anxious;Depressed|Frustrated;Frustra...,0.864121,0.003791,0.867450,0.863116,0.004094,1.8,0.5,1.3,4,97.478992,False
4,Suicidal|Depressed;Depressed|Frustrated;Frustr...,0.864087,0.004000,0.867450,0.863082,0.005503,2.3,1.0,1.3,5,96.638655,False
5,Depressed|Others;Anxious|Frustrated;Frustrated...,0.864085,0.004103,0.867450,0.863100,0.004966,2.0,0.7,1.3,6,95.798319,False
6,Depressed|Frustrated;Anxious|Frustrated;Frustr...,0.864032,0.003757,0.867383,0.863029,0.003893,1.7,0.5,1.2,7,94.957983,False
7,Suicidal|Anxious;Depressed|Others;Frustrated|O...,0.864012,0.003960,0.867383,0.863028,0.005101,2.0,0.8,1.2,8,94.117647,False
8,Suicidal|Frustrated;Depressed|Others;Frustrate...,0.864011,0.004027,0.867383,0.863028,0.005369,1.9,0.7,1.2,9,93.277311,False
9,Suicidal|Others;Depressed|Others;Frustrated|Ot...,0.864006,0.004162,0.867383,0.863038,0.005570,2.0,0.8,1.2,10,92.436975,False


PASS: control aggregation and exhaustive ranking verified


## Export and integrity verification

The final cell writes the complete Phase 10 artifact set, records frozen controls and input
provenance, verifies exact row counts and finite metrics, creates a complete SHA-256
manifest, and packages a deterministic results ZIP.

In [13]:
control_seedwise.to_csv(OUTPUT_DIR / "phase10_control_seedwise.csv", index=False)
control_summary.to_csv(OUTPUT_DIR / "phase10_control_summary.csv", index=False)
allocation_seedwise.to_csv(
    OUTPUT_DIR / "phase10_exhaustive_allocation_seedwise.csv",
    index=False,
)
allocation_summary.to_csv(
    OUTPUT_DIR / "phase10_exhaustive_allocation_summary.csv",
    index=False,
)
phase9_reproduction.to_csv(
    OUTPUT_DIR / "phase10_phase9_reproduction.csv",
    index=False,
)
threshold_sensitivity.to_csv(
    OUTPUT_DIR / "phase10_threshold_sensitivity.csv",
    index=False,
)
residual_metrics.to_csv(OUTPUT_DIR / "phase10_residual_metrics.csv", index=False)
residual_history.to_csv(
    OUTPUT_DIR / "phase10_residual_training_history.csv",
    index=False,
)
pd.DataFrame(ACCESSED_INPUTS).to_csv(
    OUTPUT_DIR / "phase10_input_inventory.csv",
    index=False,
)

write_json(
    OUTPUT_DIR / "phase10_random_nonhard_allocation.json",
    {
        "rng_seed": ALLOCATION_RNG_SEED,
        "allocation": list(RANDOM_NONHARD),
        "allocation_id": random_allocation_id,
        "candidate_count": 35,
        "test_accessed": False,
    },
)
write_json(
    OUTPUT_DIR / "phase10_lowest_bcr_allocation.json",
    {
        "allocation": list(LOWEST_BCR),
        "allocation_id": lowest_bcr_allocation_id,
        "selection": "three lowest frozen Phase 5 mean_bcr values",
        "test_accessed": False,
    },
)
write_json(
    OUTPUT_DIR / "phase10_selected_allocation_rank.json",
    {
        "allocation": list(HARD_PAIRS),
        "allocation_id": hard_allocation_id,
        "rank": int(hard_rank_row["rank"]),
        "total_allocations": 120,
        "percentile": float(hard_rank_row["percentile"]),
        "mean_macro_f1": float(hard_rank_row["mean_macro_f1"]),
        "mean_coverage": float(hard_rank_row["mean_coverage"]),
        "descriptive_only": True,
    },
)

protocol = {
    "phase": 10,
    "title": "Controls and Allocation Analysis",
    "labels": list(LABELS),
    "model_seeds": list(SEEDS),
    "H": list(HARD_PAIRS),
    "K": K_FIXED,
    "frozen_tau": FROZEN_TAU,
    "phase9_selected_mode": "specialist_routing",
    "tau_grid": TAU_GRID.tolist(),
    "allocation_rng_seed": ALLOCATION_RNG_SEED,
    "random_nonhard_allocation": list(RANDOM_NONHARD),
    "lowest_bcr_allocation": list(LOWEST_BCR),
    "allocation_count": 120,
    "control_ids": list(CONTROL_IDS),
    "A5_marker": A5_MARKER,
    "A7_marker": A7_MARKER,
    "residual_architecture": "Linear(d,h)-GELU-Linear(h,5) residual logits",
    "residual_optimizer": "AdamW",
    "residual_learning_rate": 1e-3,
    "residual_weight_decay": 1e-2,
    "residual_batch_size": 64,
    "residual_maximum_epochs": 60,
    "residual_patience": 6,
    "residual_gradient_clip": 1.0,
    "test_accessed": False,
}
assert protocol["frozen_tau"] == 0.10
assert protocol["test_accessed"] is False
write_json(OUTPUT_DIR / "phase10_protocol.json", protocol)

FINITE_METRIC_COLUMNS = {
    "control_seedwise": [
        "accuracy",
        "macro_f1",
        "balanced_accuracy",
        "routing_coverage",
        "routed_n",
        "corrections",
        "harms",
        "net_corrections",
        "macro_f1_delta_vs_A0",
    ],
    "control_summary": [
        column for column in control_summary.columns if column != "control_id"
    ],
    "allocation_seedwise": [
        "accuracy",
        "macro_f1",
        "balanced_accuracy",
        "routing_coverage",
        "routed_n",
        "corrections",
        "harms",
        "net_corrections",
    ],
    "allocation_summary": [
        column
        for column in allocation_summary.columns
        if column not in {"allocation_id", "is_hard_allocation"}
    ],
    "residual_metrics": [
        column for column in residual_metrics.columns if column != "model_seed"
    ],
    "residual_history": [
        column for column in residual_history.columns if column != "model_seed"
    ],
}
finite_frames = {
    "control_seedwise": control_seedwise,
    "control_summary": control_summary,
    "allocation_seedwise": allocation_seedwise,
    "allocation_summary": allocation_summary,
    "residual_metrics": residual_metrics,
    "residual_history": residual_history,
}
for frame_name, numeric_frame in finite_frames.items():
    numeric = numeric_frame[FINITE_METRIC_COLUMNS[frame_name]].to_numpy()
    if not np.isfinite(numeric).all():
        raise ValueError(f"Non-finite Phase 10 metric detected: {frame_name}")

forbidden_paths = [
    row["path"]
    for row in ACCESSED_INPUTS
    if "_test" in Path(row["path"]).name.lower()
    or "combined" in Path(row["path"]).name.lower()
]
assert forbidden_paths == []

execution_summary = {
    "phase": 10,
    "status": "PASS",
    "seed_count": len(SEEDS),
    "control_seedwise_rows": len(control_seedwise),
    "allocation_seedwise_rows": len(allocation_seedwise),
    "allocation_summary_rows": len(allocation_summary),
    "phase9_reproduction_pass_count": int(
        phase9_reproduction["status"].eq("PASS").sum()
    ),
    "hard_allocation_rank": int(hard_rank_row["rank"]),
    "frozen_tau": FROZEN_TAU,
    "test_accessed": False,
}
write_json(OUTPUT_DIR / "phase10_execution_summary.json", execution_summary)

report_lines = [
    "# Phase 10 Controls and Allocation Analysis Report",
    "",
    "Status: PASS",
    "",
    f"Frozen tau: {FROZEN_TAU:.2f}",
    f"Phase 9 reproductions: {len(phase9_reproduction)}/10 PASS",
    f"Frozen hard allocation rank: {int(hard_rank_row['rank'])}/120",
    "",
    "## Control summary",
    "",
    "```csv",
    control_summary.to_csv(index=False).strip(),
    "```",
    "",
    "The exhaustive allocation rank is descriptive. No test data were accessed.",
]
(OUTPUT_DIR / "phase10_report.md").write_text("\n".join(report_lines) + "\n")

manifest_rows = []
for result_path in sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file()):
    manifest_rows.append(
        {
            "path": result_path.relative_to(OUTPUT_DIR).as_posix(),
            "sha256": sha256_file(result_path),
            "bytes": result_path.stat().st_size,
        }
    )
pd.DataFrame(manifest_rows).to_csv(
    OUTPUT_DIR / "sha256_manifest.csv",
    index=False,
)
verify_manifest(OUTPUT_DIR)

zip_path = WORKING_ROOT / "MentalDistress_Phase10_Controls_and_Allocation_Results.zip"
with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as archive:
    for result_path in sorted(
        path for path in OUTPUT_DIR.rglob("*") if path.is_file()
    ):
        archive_name = (
            OUTPUT_DIR.name
            + "/"
            + result_path.relative_to(OUTPUT_DIR).as_posix()
        )
        archive_info = zipfile.ZipInfo(archive_name, (2020, 1, 1, 0, 0, 0))
        archive_info.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(archive_info, result_path.read_bytes())

print("=" * 76)
print("PHASE 10 COMPLETE")
print("Frozen tau:", FROZEN_TAU)
print("Hard allocation rank:", int(hard_rank_row["rank"]), "/ 120")
print("Results ZIP:", zip_path)
print("ZIP SHA256:", sha256_file(zip_path))
print("Test data accessed: False")
print("=" * 76)

PHASE 10 COMPLETE
Frozen tau: 0.1
Hard allocation rank: 3 / 120
Results ZIP: /kaggle/working/MentalDistress_Phase10_Controls_and_Allocation_Results.zip
ZIP SHA256: 7d9e0065632e024d4f9aece380a72cefec2b2bc3e03e18bf51e076229a421ff3
Test data accessed: False
